In [ ]:
import boto3
import sagemaker
import json
import numpy as np
from sagemaker.xgboost import XGBoostProcessor
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.parameters import ParameterFloat
from sagemaker.workflow.steps import ProcessingStep, TrainingStep, TuningStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.workflow.model_step import ModelStep
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.xgboost import XGBoost
from sagemaker.tuner import ContinuousParameter, IntegerParameter
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.functions import Join
from sagemaker.tuner import (
    HyperparameterTuner,
    ContinuousParameter,
    IntegerParameter,
    CategoricalParameter
)
from sagemaker.workflow.lambda_step import LambdaStep, LambdaOutput, LambdaOutputTypeEnum
from sagemaker.lambda_helper import Lambda
from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.processing import ScriptProcessor
from sagemaker.workflow.check_job_config import CheckJobConfig
from sagemaker.model_monitor import DataCaptureConfig
from sagemaker.workflow.lambda_step import LambdaOutput, LambdaOutputTypeEnum


sagemaker_client = boto3.client("sagemaker")
cloudwatch_client = boto3.client("cloudwatch")

deploy_lambda_arn = "arn:aws:lambda:eu-north-1:861976376325:function:healthcare-severity-classification-model"
create_model_package_lambda_arn = "arn:aws:lambda:eu-north-1:861976376325:function:severity-classification-model-group"
# Initialize
pipeline_session = PipelineSession()
bucket = "saran-healthcare-severity-data"
role = sagemaker.get_execution_role()
region = sagemaker.Session().boto_region_name
script_dir = "/home/sagemaker-user/scripts/"
monitoring_s3_path = f"s3://{bucket}/healthcare-data/monitoring/data-capture/"

# Parameters
accuracy_threshold = ParameterFloat(name="AccuracyThreshold", default_value=0.50)

# Preprocessing Step
sklearn_processor = SKLearnProcessor(
    framework_version='1.2-1',
    role=role,
    instance_type="ml.t3.medium",
    instance_count=1,
    sagemaker_session=pipeline_session
)

step_preprocess = ProcessingStep(
    name="PreprocessData",
    processor=sklearn_processor,
    inputs=[
        sagemaker.processing.ProcessingInput(
            source=f"s3://{bucket}/healthcare-data/raw-data",
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        sagemaker.processing.ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/train",
            destination=f"s3://{bucket}/healthcare-data/preprocessed/train"
        ),
        sagemaker.processing.ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/test",
            destination=f"s3://{bucket}/healthcare-data/preprocessed/test"
        )
    ],
    code=f"{script_dir}/preprocessing.py"
)


# Hyperparameter Tuning Configuration
xgb_classifier = XGBoost(
    entry_point=f"{script_dir}/train.py",
    framework_version="1.7-1",
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    output_path=f"s3://{bucket}/healthcare-data/model",
    hyperparameters={
        "num_class": 3,
        "objective": "multi:softprob",
        "eval_metric": ["merror", "mlogloss"],
        "tree_method": "hist",
        "early_stopping_rounds": 20
    },
    sagemaker_session=pipeline_session
    #use_spot_instances=True,
    #max_wait=3600,
    #max_run=1800
)


tuner = HyperparameterTuner(
    estimator=xgb_classifier,
    objective_metric_name="validation:merror", 
    objective_type="Minimize",
    hyperparameter_ranges={
        "max_depth": IntegerParameter(3, 10),
        "eta": ContinuousParameter(0.01, 0.3),
        "min_child_weight": IntegerParameter(1, 10),
        "subsample": ContinuousParameter(0.5, 1),
        "gamma": ContinuousParameter(0, 5),
        "alpha": ContinuousParameter(0, 5),
        "lambda": ContinuousParameter(0, 5)
    },
    metric_definitions=[
        {
            "Name": "validation:merror",
            # Match XGBoost's output with dataset index (validation_0-merror)
            "Regex": r"(?:\[(\d+)\]\svalidation_0-merror:|final-validation_0-merror:)(\d+\.\d+)"
        }
    ],
    #max_jobs=10,
    #max_parallel_jobs=2,
    early_stopping_type="Auto",
    strategy="Bayesian"
)


step_tune = TuningStep(
    name="HyperparameterTuning",
    tuner=tuner,
    inputs={
        "train": TrainingInput(
            s3_data=step_preprocess.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv"
        ),
        "validation": TrainingInput(
            s3_data=step_preprocess.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            content_type="text/csv"
        )
    }
)

# **Model Monitoring: Use ScriptProcessor instead of DefaultModelMonitor**
baseline_processor = ScriptProcessor(
    image_uri=sagemaker.image_uris.retrieve("xgboost", region, "1.7-1"),
    command=["python3"],
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    sagemaker_session=pipeline_session
)

monitoring_processor = ScriptProcessor(
    image_uri=sagemaker.image_uris.retrieve("xgboost", region, "1.7-1"),
    command=["python3"],
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    sagemaker_session=pipeline_session,
    env={"SM_LOG_LEVEL": "INFO"}
)

# **Baseline Creation Step (Runs Once Before Deployment)**
step_create_baseline = ProcessingStep(
    name="CreateBaseline",
    processor=baseline_processor,
    inputs=[
        ProcessingInput(
            source=f"s3://{bucket}/healthcare-data/preprocessed/train",
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/output",
            destination=f"s3://{bucket}/healthcare-data/monitoring/baseline/"
        )
    ],
    code=f"{script_dir}/generate_baseline.py",
    depends_on=[step_preprocess, step_tune]
)

# Evaluation Step
evaluation_processor = sagemaker.processing.ScriptProcessor(
    image_uri=sagemaker.image_uris.retrieve("xgboost", region, "1.7-1"),
    command=["python3"],
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    sagemaker_session=pipeline_session
)

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json"
)

step_evaluate = ProcessingStep(
    name="EvaluationStep",
    processor=evaluation_processor,
    inputs=[
        # Use built-in method to get best model path
        ProcessingInput(
            source=step_tune.get_top_model_s3_uri(
                top_k=0,  # Get best model
                s3_bucket=bucket,
                prefix="healthcare-data/model"
            ),
            destination="/opt/ml/processing/model",
            s3_data_distribution_type="FullyReplicated"
        ),
        # Test data from preprocessing step
        ProcessingInput(
            source=step_preprocess.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/test"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation",
            destination=f"s3://{bucket}/healthcare-data/evaluation"
        )
    ],
    code=f"{script_dir}/evaluation.py",
    property_files=[evaluation_report]
)

# Condition Step and Model Registration
condition = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=step_evaluate.name,
        property_file=evaluation_report,
        json_path="accuracy"
    ),
    right=accuracy_threshold
)

model = sagemaker.model.Model(
    image_uri=xgb_classifier.training_image_uri(),
    model_data=step_tune.get_top_model_s3_uri(top_k=0, s3_bucket=bucket, prefix="healthcare-data/model"),
    role=role,
    sagemaker_session=pipeline_session
)

step_create_model_package_group = LambdaStep(
    name="CreateModelPackageGroup",
    lambda_func=Lambda(create_model_package_lambda_arn),
    inputs={"model_package_group_name": "SeverityClassificationlt"},
    outputs=[]  # No outputs needed, just ensures the group exists
)

step_register = ModelStep(
    name="RegisterModel",
    step_args=model.register(
        content_types=["text/csv"],
        response_types=["text/csv"],
        inference_instances=["ml.m5.xlarge"],
        model_package_group_name="SeverityClassificationlt"
    ),
    depends_on=[step_create_model_package_group]
)

approve_step = LambdaStep(
    name="ApproveModel",
    lambda_func=Lambda("arn:aws:lambda:eu-north-1:861976376325:function:severitiy-classification-approval-model-registry"),
    inputs={
        "model_package_arn": step_register.properties.ModelPackageArn
    },
    depends_on=[step_register]
)


# 2. Create Lambda Step for Deployment
lambda_deploy_step = LambdaStep(
    name="DeployModelEndpoint",
    lambda_func=Lambda(deploy_lambda_arn),
    inputs={
        "model_package_arn": step_register.properties.ModelPackageArn,
        "endpoint_name": "health-severity-endpoint",
        "role_arn": role,
        "instance_type": "ml.m5.xlarge",
        "initial_instance_count": 1,
        "region": region
    },
    outputs=[
        LambdaOutput(
            output_name="statusCode", 
            output_type=LambdaOutputTypeEnum.String
        ),
        LambdaOutput(
            output_name="body", 
            output_type=LambdaOutputTypeEnum.String
        )
    ],
    depends_on=[approve_step]
)

# **Enable Data Capture for Model Monitoring**
data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=monitoring_s3_path,
    capture_options=["REQUEST", "RESPONSE"],
)

#check_monitoring_data = CheckJobConfig(
#    check_type="S3ObjectExists",
#    s3_uri=f"s3://{bucket}/healthcare-data/monitoring/data-capture/"
#)

# **Model Monitoring: Use ScriptProcessor**
baseline_processor = ScriptProcessor(
    image_uri=sagemaker.image_uris.retrieve("xgboost", region, "1.7-1"),
    command=["python3"],
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    sagemaker_session=pipeline_session
)

# **Step to Enable Data Capture on Endpoint**
step_enable_data_capture = LambdaStep(
    name="EnableDataCapture",
    lambda_func=Lambda("arn:aws:lambda:eu-north-1:861976376325:function:severity-enable-data-capture"),
    inputs={
        "endpoint_name": "health-severity-endpoint",
        "data_capture_config": json.dumps({
            "EnableCapture": True,
            "InitialSamplingPercentage": 100,
            "DestinationS3Uri": f"s3://{bucket}/healthcare-data/monitoring/data-capture/",
            "CaptureOptions": [
                {"CaptureMode": "Input"},
                {"CaptureMode": "Output"}
            ],
            "CaptureContentTypeHeader": {
                "CsvContentTypes": ["text/csv"],
                "JsonContentTypes": ["application/json"]
            }
        })
    },
    depends_on=[lambda_deploy_step]
)


# **Step to Check if Data Exists in S3 Before Monitoring**
step_check_data = LambdaStep(
    name="CheckMonitoringData",
    lambda_func=Lambda("arn:aws:lambda:eu-north-1:861976376325:function:severtiy-check-s3-data"),
    inputs={"s3_path": monitoring_s3_path},
    outputs=[
        LambdaOutput(
            output_name="DataAvailable",
            output_type=LambdaOutputTypeEnum.Boolean
        )
    ],
    depends_on=[step_enable_data_capture]
)

# **Monitoring Step (Runs on a Schedule)**
step_monitoring = ProcessingStep(
    name="MonitorModel",
    processor=monitoring_processor,
    depends_on=["DeployModelEndpoint", "EnableDataCapture", "CheckMonitoringData"],  # Ensures Monitoring runs only after data is captured
    inputs=[
        ProcessingInput(
            source="s3://saran-healthcare-severity-data/healthcare-data/monitoring/data-capture/health-severity-endpoint/primary/",
            destination="/opt/ml/processing/input"
        ),
        ProcessingInput(
            source=f"s3://{bucket}/healthcare-data/monitoring/baseline/",
            destination="/opt/ml/processing/baseline"
        )
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/output",
            destination=f"s3://{bucket}/healthcare-data/monitoring/reports/"
        )
    ],
    code=f"{script_dir}/monitoring.py"
)

step_condition = ConditionStep(
    name="CheckAccuracy",
    conditions=[condition],
    # if_steps=[step_register, approve_step, lambda_deploy_step], 
    if_steps=[step_register],
    else_steps=[]
)

# Assemble Pipeline
pipeline = Pipeline(
    name="SeverityClassificationPipelinelt",
    parameters=[accuracy_threshold],
    steps=[step_preprocess, step_create_baseline, step_tune, step_evaluate, step_condition, approve_step, lambda_deploy_step, step_enable_data_capture, step_check_data, step_monitoring]
)

pipeline.upsert(role_arn=role)
execution = pipeline.start()
#execution.describe()